# Credit Score — Adversarial Robustness & Model Security

Camada de segurança sobre o modelo de risco de crédito. O modelo de produção é um XGBoost no SageMaker; aqui usamos um **proxy local** (`HistGradientBoostingClassifier` sobre o mesmo schema UCI de 23 features) para estudar:

- evasão tabular com **feature constraints** (só o que o solicitante consegue manipular);
- curvas de robustez (fração de decisões revertíveis por orçamento de perturbação);
- adversarial training tabular;
- model extraction por consulta.

`train → validate → attack → measure robustness → mitigate → monitor`, não só `train → accuracy → deploy`.

## Setup

In [ ]:
import sys, os
sys.path.append(os.path.abspath('.'))
import numpy as np
from src.adversarial import (load_dataset, split, CreditRiskModel, minimal_evasion,
                             flip_costs, robustness_curve, adversarially_train,
                             model_extraction_attack)

# usa UCI_Credit_Card.csv se existir na raiz; senão, dataset sintético com o mesmo schema
df = load_dataset()
Xtr, ytr, Xte, yte = split(df)
model = CreditRiskModel().fit(Xtr, ytr)
print('acurácia teste:', (model.predict(Xte) == yte).mean())


## 1. Evasão de um solicitante high_risk

A busca só mexe nas features que um solicitante realmente controla (`PAY_AMT*`, `BILL_AMT*` recentes), dentro de limites. `SEX`, `AGE`, `EDUCATION`, `MARRIAGE`, `LIMIT_BAL` e o histórico de atraso (`PAY_*`) ficam fixos.

In [ ]:
high_risk = Xte[model.predict(Xte) == 1]
x0 = high_risk[0]
print('decisão original:', model.decision(x0), '| score:', round(model.score(x0)[0], 3))

res = minimal_evasion(model, x0, seed=0)
if res['success'] and not res.get('already_low_risk'):
    print('decisão após evasão:', model.decision(res['x_adv']),
          '| score:', round(res['final_score'], 3),
          '| custo L1 normalizado:', round(res['cost'], 3))
    from src.adversarial.dataset import FEATURES
    for i, f in enumerate(FEATURES):
        if not np.isclose(res['x_adv'][i], x0[i]):
            print(f'  {f}: {x0[i]:.0f} -> {res["x_adv"][i]:.0f}')
else:
    print('não foi possível reverter dentro dos limites de perturbação')


## 2. Curva de robustez

Fração dos solicitantes high_risk cuja decisão pode ser revertida com custo de perturbação até B. Curva que sobe rápido = fronteira frágil/manipulável.

In [ ]:
costs = flip_costs(model, Xte, max_samples=40, seed=0)
for pt in robustness_curve(costs):
    print(f"budget {pt['budget']:<5} -> {pt['flippable_fraction']:.1%} revertíveis")


## 3. Adversarial training

Retreina incluindo versões evadidas (ainda rotuladas como alto risco). Objetivo: a mesma perturbação pequena não muda mais a decisão.

In [ ]:
hardened = adversarially_train(Xtr, ytr, base_model=model, max_adv=60, seed=0)
costs_after = flip_costs(hardened, Xte, max_samples=40, seed=0)
print('revertíveis com budget 0.25  — antes :', np.mean(costs <= 0.25))
print('revertíveis com budget 0.25  — depois:', np.mean(costs_after <= 0.25))
print('acurácia limpa depois:', (hardened.predict(Xte) == yte).mean())


## 4. Model extraction

Substituto treinado só com as decisões do modelo-alvo. Fidelity alta = o modelo pode ser reconstruído por quem só tem a API.

In [ ]:
ext = model_extraction_attack(model, Xtr, Xte, yte)
print(ext)


## Nota

Este módulo integra o **robustness gate** do Argus (`ml-platform/adversarial-evaluation/`) via o `ModelSecurityReport` do ThemisAI — um modelo de crédito com fronteira muito manipulável não é promovido a produção.